# 04 — Job Clustering, Visualization & Skill-Gap Report
SmartHire Phase 2 — Discovery + Insight.

Covers: K-Means clustering on job vectors, elbow + silhouette sweep, PCA scatter plot, cluster labelling, and the skill-gap report for a sample resume.

**Prerequisites:** `python -m src.models.recommender` must have run first (needs `models/job_vectors.pkl`).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from src.config import FIGURES_DIR
from src.models.recommender import load_job_corpus, load_job_vectors
from src.models.clustering import (
    sweep_k, pick_best_k, fit_clustering, save_clustering,
    load_clustering, get_2d_projection,
)
from src.models.skill_gap import generate_skill_gap_report, format_report_for_display

sns.set_style('whitegrid')

## 1. Load data

In [ ]:
job_corpus = load_job_corpus()
job_vectors, job_vectorizer = load_job_vectors()
print(f'{len(job_corpus)} jobs | {job_vectors.shape}')

## 2. Elbow + Silhouette sweep
Runs K-Means for k = 4..15 and records inertia (elbow method) and silhouette score.

In [ ]:
sweep_df = sweep_k(job_vectors)
best_k = pick_best_k(sweep_df)
sweep_df

## 3. Elbow curve + silhouette plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(sweep_df['k'], sweep_df['inertia'], 'o-', color='steelblue')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method')

ax2.plot(sweep_df['k'], sweep_df['silhouette'], 's-', color='darkorange')
ax2.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs k')
ax2.legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'clustering_elbow_silhouette.png', dpi=150)
plt.show()
print(f'Chosen k = {best_k}')

## 4. Fit final K-Means and save

In [ ]:
km, svd, labels = fit_clustering(job_vectors, k=best_k)
save_clustering(km, svd, labels)

job_corpus['cluster'] = labels
print(job_corpus.groupby('cluster')['title'].count().rename('jobs_in_cluster'))

## 5. PCA 2-D scatter plot of job clusters

In [ ]:
coords = get_2d_projection(job_vectors, svd)
colors = cm.tab20(np.linspace(0, 1, best_k))

fig, ax = plt.subplots(figsize=(10, 7))
for k_id in range(best_k):
    mask = labels == k_id
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=[colors[k_id]], label=f'Cluster {k_id}', alpha=0.6, s=15)
ax.set_title('Job Clusters — PCA 2-D Projection')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(markerscale=2, fontsize=8, ncol=2)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'clustering_pca_scatter.png', dpi=150)
plt.show()

## 6. Label the clusters — inspect top jobs per cluster
Manually assign a descriptive label to each cluster by looking at the most frequent job titles in it.

In [ ]:
for k_id in range(best_k):
    titles = job_corpus.loc[job_corpus['cluster'] == k_id, 'title']
    print(f'Cluster {k_id} ({len(titles)} jobs):')
    print('  ', titles.value_counts().head(5).to_dict())
    print()

## 7. Skill-Gap Report — demo
Given a sample resume and its target cluster (e.g. the Data Science cluster), compute which skills to add.

In [ ]:
SAMPLE_RESUME = """
Python developer with 2 years experience. Familiar with pandas and basic SQL.
Some exposure to matplotlib for visualizations. Looking to move into data science roles.
"""

# Replace TARGET_CLUSTER with the cluster ID that corresponds to Data Science
# after you inspect the cluster labels in Section 6 above.
TARGET_CLUSTER = 0   # <-- update this after inspecting Section 6

report = generate_skill_gap_report(
    resume_raw_text=SAMPLE_RESUME,
    cluster_id=TARGET_CLUSTER,
    job_corpus=job_corpus,
    job_vectors=job_vectors,
    cluster_labels=labels,
    vectorizer=job_vectorizer,
)

print(format_report_for_display(report))

## 8. Visualise the skill gap as a bar chart

In [ ]:
gap = report['gap']
skill_data = (
    [(s, 'Present') for s in gap['matched']] +
    [(s, 'Missing') for s in gap['missing'][:15]]
)
skill_df = pd.DataFrame(skill_data, columns=['skill', 'status'])

fig, ax = plt.subplots(figsize=(9, max(4, len(skill_df) * 0.35)))
colors = skill_df['status'].map({'Present': '#2ecc71', 'Missing': '#e74c3c'})
ax.barh(skill_df['skill'], [1] * len(skill_df), color=colors)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2ecc71', label='Present in resume'),
    Patch(color='#e74c3c', label='Missing — add to CV'),
])
ax.set_title(f'Skill-Gap Report for Cluster {TARGET_CLUSTER}  (match: {gap["match_pct"]}%)')
ax.set_xlabel('')
ax.tick_params(left=True)
ax.get_xaxis().set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / f'skill_gap_cluster_{TARGET_CLUSTER}.png', dpi=150)
plt.show()

## 9. Notes / findings
- Chosen k and rationale (elbow vs silhouette): ...
- Descriptive label assigned to each cluster: ...
- Quality of skill-gap report (are missing skills genuinely useful?): ...
- Ideas for improvement (LDA/NMF topic modeling, bigram skill phrases): ...